# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`

This notebook guides you through loading, exploring, and processing the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load the metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the metadata object
metadata = dataset.metadata.to_json()

print(f"Dataset title: {metadata['name']}")
print(f"Description: {metadata['description']}")

## 2. Data Overview

Review available record sets, fields, and their IDs.

The Croissant dataset may contain multiple record sets, each uniquely identified by their `@id`.


In [ ]:
# List available record sets by @id (using metadata)
if hasattr(dataset.metadata, 'record_set'):
    record_sets = dataset.metadata.record_set
elif 'recordSet' in metadata:
    record_sets = metadata['recordSet']
else:
    record_sets = []

if not record_sets:
    # Attempt to fetch record sets from loaded dataset
    try:
        # mlcroissant exposes dataset.record_sets property for Croissant v1.0
        record_sets = dataset.record_sets
        print("Record sets loaded from Dataset.record_sets property:")
    except AttributeError:
        record_sets = []
        print("No record sets found in metadata or dataset object.")

# Print available record sets (with @id)
if isinstance(record_sets, dict):
    # If it's a dictionary, print the keys as @id
    for record_set_id, record_set in record_sets.items():
        print(f"Record Set @id: {record_set_id}\nTitle: {record_set.get('name', '')}\nDescription: {record_set.get('description', '')}\n")
        # Show the available fields in this record set
        if 'fields' in record_set:
            print("Fields:")
            for f in record_set['fields']:
                print(f"  Field @id: {f.get('@id', str(f))}")
    record_set_ids = list(record_sets.keys())
elif isinstance(record_sets, list):
    # If it's a list, print @id and name for each
    record_set_ids = []
    for rset in record_sets:
        rs_id = rset.get('@id', rset)
        rs_name = rset.get('name', '')
        rs_desc = rset.get('description', '')
        record_set_ids.append(rs_id)
        print(f"Record Set @id: {rs_id}\nTitle: {rs_name}\nDescription: {rs_desc}\n")
        if 'fields' in rset:
            print("Fields:")
            for fld in rset['fields']:
                print(f"  Field @id: {fld.get('@id', str(fld))}")
else:
    # As fallback, print the type
    print(f"Record sets type: {type(record_sets)}")
    record_set_ids = []

# If no record sets found, try to infer available @id from API
if not record_set_ids:
    try:
        record_set_ids = list(dataset.record_sets.keys())
        print(f"Available Record Set @ids: {record_set_ids}")
    except AttributeError:
        print("Could not infer record sets from dataset object.")

# Let's print sample records for each record set
for record_set_id in record_set_ids:
    print(f"\nSample records from Record Set @id: {record_set_id}")
    try:
        for i, record in enumerate(dataset.records(record_set=record_set_id)):
            print(record)
            if i >= 2:  # Show up to 3 records
                break
    except Exception as e:
        print(f"Unable to load records for record set {record_set_id}: {str(e)}")

## 3. Data Extraction

Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

Refer to the `@id` of each record set and each field when extracting and analyzing data with `mlcroissant`.

In [ ]:
# Extract data from each record set and load into pandas DataFrames

# Use the list of @id obtained above
dataframes = {}

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for Record Set @id: {rs_id}")
        print(f"Fields/Columns (@id): {df.columns.tolist()}")
        print(df.head())
    except Exception as e:
        print(f"Failed to load DataFrame for record set {rs_id}: {str(e)}")
        dataframes[rs_id] = None


## 4. Exploratory Data Analysis (EDA)

Apply common processing steps: filtering records based on criteria, normalizing numeric fields, and grouping by key attributes.

Below, we'll select a numeric field and perform example filtering, normalization, and grouping. You can modify the field `@id` to match your interest from the printed DataFrame columns above.

In [ ]:
# Choose a record set to analyze
# Example: Use first available record set
if record_set_ids:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
else:
    record_set_id = None
    df = None

# Select numeric field @id from this record set
# (Replace with appropriate field @id as needed)
if df is not None:
    numeric_field_id = None
    # Find first field with numeric dtype
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

    if numeric_field_id is None:
        print("No numeric field found for EDA.")
    else:
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with '{numeric_field_id}' > {threshold}:")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}' for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping: Try to group by any categorical field
        group_field_id = None
        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id:
                group_field_id = col
                break

        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped records by '{group_field_id}':")
            print(grouped_df.head())
        else:
            print("No suitable categorical field for grouping found.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset using matplotlib or seaborn.

Below is an example for plotting the normalized numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field distribution (if available)
if df is not None and numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of '{numeric_field_id}' in Record Set {record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If normalized field was computed above
    norm_col = f"{numeric_field_id}_normalized"
    if norm_col in filtered_df.columns:
        plt.figure(figsize=(8, 4))
        sns.histplot(filtered_df[norm_col], kde=True)
        plt.title(f"Normalized '{numeric_field_id}' Distribution (Filtered)")
        plt.xlabel(norm_col)
        plt.ylabel("Count")
        plt.show()


## 6. Conclusion

- This notebook demonstrated loading and exploration of the FAIR^2 clinical dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.
- The dataset is structured according to the Croissant schema, and all record sets, fields, and columns are referenced by their `@id` for reproducible analysis.
- You can use the provided DataFrame and field references to further explore genotype-phenotype correlations, clinical features, hormone levels, and genetic variants for non-classical 21-hydroxylase deficiency.
- Due to the limited sample size and clinical setting, use caution when generalizing findings beyond the dataset scope.
